In [1]:
import os
import speech_recognition as sr
from moviepy.editor import VideoFileClip
import pandas as pd
import json
from pathlib import Path

In [3]:
class VideoTranscriptExtractor:
    def __init__(self, folder_path):
        """
        Initialize the transcript extractor
        
        Args:
            folder_path (str): Path to folder containing videos
        """
        self.folder_path = folder_path
        self.recognizer = sr.Recognizer()
        self.supported_formats = ['.mp4', '.avi', '.mov', '.mkv', '.wmv', '.flv']
        
    def extract_id_emotion(self, filename):
        """
        Extract ID and emotion from filename like '1_anger.mp4'
        
        Args:
            filename (str): Video filename
            
        Returns:
            tuple: (id, emotion)
        """
        name_without_ext = os.path.splitext(filename)[0]
        parts = name_without_ext.split('_', 1)  # Split hanya pada underscore pertama
        
        if len(parts) == 2:
            return parts[0], parts[1]
        else:
            return filename, "unknown"
    
    def get_video_duration(self, video_path):
        """
        Get video duration in seconds
        
        Args:
            video_path (str): Path to video file
            
        Returns:
            float: Duration in seconds
        """
        try:
            video = VideoFileClip(video_path)
            duration = video.duration
            video.close()
            return duration
        except Exception as e:
            print(f"Error getting duration for {video_path}: {str(e)}")
            return 0
    
    def format_duration(self, seconds):
        """
        Format duration from seconds to MM.SS format
        
        Args:
            seconds (float): Duration in seconds
            
        Returns:
            str: Formatted duration (e.g., "1.40" for 1 minute 40 seconds)
        """
        minutes = int(seconds // 60)
        remaining_seconds = int(seconds % 60)
        return f"{minutes}.{remaining_seconds:02d}"
    
    def video_to_audio(self, video_path, audio_path):
        """
        Convert video to audio file
        
        Args:
            video_path (str): Path to video file
            audio_path (str): Path for output audio file
        """
        try:
            video = VideoFileClip(video_path)
            video.audio.write_audiofile(audio_path, verbose=False, logger=None)
            video.close()
            return True
        except Exception as e:
            print(f"Error converting {video_path} to audio: {str(e)}")
            return False
    
    def detect_language_and_transcribe(self, audio_path):
        """
        Detect language and transcribe audio to text
        Tries both Indonesian and English
        
        Args:
            audio_path (str): Path to audio file
            
        Returns:
            str: Transcribed text
        """
        try:
            with sr.AudioFile(audio_path) as source:
                # Adjust for ambient noise
                self.recognizer.adjust_for_ambient_noise(source, duration=1)
                audio_data = self.recognizer.record(source)
            
            # Language codes to try
            languages = ['id-ID', 'en-US']
            
            for lang in languages:
                try:
                    text = self.recognizer.recognize_google(audio_data, language=lang)
                    if text and text.strip():
                        return text.strip()
                except sr.UnknownValueError:
                    continue
                except sr.RequestError as e:
                    print(f"Google Speech Recognition error for {lang}: {e}")
                    continue
            
            # If Google fails, try offline recognition
            try:
                text = self.recognizer.recognize_sphinx(audio_data)
                return text.strip() if text else "Could not understand audio"
            except:
                return "Could not understand audio"
                    
        except Exception as e:
            return f"Error processing audio: {str(e)}"
    
    def process_videos(self):
        """
        Process all videos in the folder and extract transcripts with duration
        
        Returns:
            list: List of dictionaries containing video info and transcripts
        """
        results = []
        temp_audio_dir = "temp_audio"
        
        # Create temporary directory for audio files
        os.makedirs(temp_audio_dir, exist_ok=True)
        
        # Get all video files
        video_files = [f for f in os.listdir(self.folder_path) 
                      if any(f.lower().endswith(ext) for ext in self.supported_formats)]
        
        total_files = len(video_files)
        print(f"Found {total_files} video files to process...")
        
        for i, filename in enumerate(video_files, 1):
            print(f"Processing {i}/{total_files}: {filename}")
            
            # Extract ID and emotion
            video_id, emotion = self.extract_id_emotion(filename)
            
            # Paths
            video_path = os.path.join(self.folder_path, filename)
            audio_path = os.path.join(temp_audio_dir, f"{video_id}_{emotion}.wav")
            
            # Get video duration
            duration_seconds = self.get_video_duration(video_path)
            duration_formatted = self.format_duration(duration_seconds)
            
            # Convert video to audio
            if self.video_to_audio(video_path, audio_path):
                # Extract transcript with language detection
                transcript = self.detect_language_and_transcribe(audio_path)
                
                # Clean up temporary audio file
                try:
                    os.remove(audio_path)
                except:
                    pass
            else:
                transcript = "Error: Could not extract audio from video"
            
            # Store result in the exact format requested
            result = {
                'id': video_id,
                'text': transcript,
                'durasi': duration_formatted,
                'emotion': emotion
            }
            results.append(result)
            
            print(f"✓ {filename} -> Duration: {duration_formatted}, Text: {transcript[:50]}...")
        
        # Clean up temporary directory
        try:
            os.rmdir(temp_audio_dir)
        except:
            pass
            
        return results
    
    def save_to_csv(self, results, output_filename="video_transcripts.csv"):
        """
        Save results to CSV file in the exact format requested
        
        Args:
            results (list): List of transcript results
            output_filename (str): Output CSV filename
        """
        # Create DataFrame with exact column order: id, text, durasi, emotion
        df = pd.DataFrame(results, columns=['id', 'text', 'durasi', 'emotion'])
        df.to_csv(output_filename, index=False, encoding='utf-8')
        
        print(f"Results saved to: {output_filename}")
        
        # Display preview
        print("\nPreview hasil (5 baris pertama):")
        print(df.head().to_string(index=False))


def main():
    # Konfigurasi - GANTI PATH INI SESUAI FOLDER VIDEO ANDA
    FOLDER_PATH = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\video"  # Ganti dengan path folder video Anda
    OUTPUT_FILENAME = "video_transcripts.csv"   # Nama file output
    
    # Pastikan folder exists
    if not os.path.exists(FOLDER_PATH):
        print(f"Error: Folder {FOLDER_PATH} tidak ditemukan!")
        print("Silakan ubah FOLDER_PATH pada kode dengan path yang benar.")
        return
    
    # Initialize extractor
    extractor = VideoTranscriptExtractor(FOLDER_PATH)
    
    # Process videos
    print("Memulai ekstraksi transkrip dan durasi...")
    print("Sistem akan otomatis mendeteksi bahasa Indonesia dan Inggris.")
    print("-" * 60)
    
    results = extractor.process_videos()
    
    # Save results to CSV
    extractor.save_to_csv(results, OUTPUT_FILENAME)
    
    print(f"\n🎉 Selesai! {len(results)} video berhasil diproses.")
    print(f"📄 File output: {OUTPUT_FILENAME}")


# Contoh penggunaan langsung
def process_folder_directly(folder_path, output_file="hasil_transkrip.csv"):
    """
    Fungsi untuk memproses folder secara langsung
    
    Args:
        folder_path (str): Path ke folder video
        output_file (str): Nama file output CSV
    """
    extractor = VideoTranscriptExtractor(folder_path)
    results = extractor.process_videos()
    extractor.save_to_csv(results, output_file)
    return results


if __name__ == "__main__":
    main()

Memulai ekstraksi transkrip dan durasi...
Sistem akan otomatis mendeteksi bahasa Indonesia dan Inggris.
------------------------------------------------------------
Found 776 video files to process...
Processing 1/776: 100_Proud.mp4
✓ 100_Proud.mp4 -> Duration: 0.51, Text: we can do you have to make time to reflect what is...
Processing 2/776: 101_Trust.mp4
✓ 101_Trust.mp4 -> Duration: 0.51, Text: we can do you have to make time to reflect what is...
Processing 3/776: 102_Surprise.mp4
✓ 102_Surprise.mp4 -> Duration: 0.51, Text: we can do you have to make time to reflect what is...
Processing 4/776: 103_Surprise.mp4
✓ 103_Surprise.mp4 -> Duration: 0.57, Text: kekuatan otot kaki kalian dengan cara duduk berdir...
Processing 5/776: 104_Sadness.mp4
✓ 104_Sadness.mp4 -> Duration: 0.54, Text: orang wajib yang perlu aku bawa pas lagi raket sel...
Processing 6/776: 105_Proud.mp4
Google Speech Recognition error for id-ID: recognition request failed: Bad Request
Google Speech Recognition error f